## EDA

In [13]:
pip install shap

   ---------------------------------------- 0.0/555.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/555.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/555.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/555.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/555.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/555.9 kB ? eta -:--:--
   ------------------ --------------------- 262.1/555.9 kB ? eta -:--:--
   ------------------ --------------------- 262.1/555.9 kB ? eta -:--:--
   ------------------ --------------------- 262.1/555.9 kB ? eta -:--:--
   ------------------ --------------------- 262.1/555.9 kB ? eta -:--:--
   ------------------ --------------------- 262.1/555.9 kB ? eta -:--:--
   -------------------------------------- 555.9/555.9 kB 281.1 kB/s eta 0:00:00
   ---------------------------------------- 0.0/38.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.1 MB ? e

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

Посмотрим на пропуски в датасетах 


In [2]:
def scan_anomalies(
    base_df: pd.DataFrame,
    cnt_df: pd.DataFrame,
    min_valid_price: int = 50000,
    max_price_quantile: float = 0.999,
    max_mileage_quantile: float = 0.995,
    min_valid_mileage: int = 100,
    mileage_grace_years: int = 3,
    max_overprice_ratio: float = 2.0,
    bot_traffic_limit: int = 150
) -> None:
    """
    Сканирует датафреймы на наличие аномалий, используя статистические перцентили
    и вынесенные бизнес-правила.

    Args:
        base_df: Исходный датафрейм с характеристиками автомобилей.
        cnt_df: Датафрейм с историей контактов.
        min_valid_price: Минимально адекватная цена авто (отсечение металлолома).
        max_price_quantile: Верхний квантиль для цены (отсечение выбросов).
        max_mileage_quantile: Верхний квантиль для пробега (отсечение ошибок ввода).
        min_valid_mileage: Минимально возможный реальный пробег для старого авто.
        mileage_grace_years: Возраст авто (в годах), до которого пробег может быть нулевым.
        max_overprice_ratio: Порог завышения цены относительно рынка (2.0 = 200% оверпрайс).
        bot_traffic_limit: Лимит для отсечения аномальных всплесков (ботов) в логах.
    """
    print("=== АНАЛИЗ ПРОПУСКОВ (NaN) В БАЗЕ ===")

    missing_stats = base_df.isna().sum()
    missing_percent = (missing_stats / len(base_df)) * 100
    missing_df = pd.DataFrame({'Пропуски': missing_stats, 'Доля (%)': missing_percent})
    print(missing_df[missing_df['Пропуски'] > 0].sort_values(by='Доля (%)', ascending=False).round(2))

    print("-" * 50)

    print("\n=== ДЕТЕКТОР АНОМАЛИЙ В BASE_DF ===")

    # 1. Аномалии цены (Бизнес-минимум + Статистический максимум)
    mask_price_low = base_df['price'] < min_valid_price
    price_upper_bound = base_df['price'].quantile(max_price_quantile)
    mask_price_high = base_df['price'] > price_upper_bound

    print(f"Авто дешевле {min_valid_price:,.0f} руб: {mask_price_low.sum()} шт.")
    print(f"Авто дороже {price_upper_bound:,.0f} руб ({max_price_quantile*100}% квантиль): {mask_price_high.sum()} шт.")

    # 2. Аномалии пробега (Статистический максимум + Бизнес-минимум)
    mileage_upper_bound = base_df['mileage'].quantile(max_mileage_quantile)
    mask_mileage_high = base_df['mileage'] > mileage_upper_bound

    current_year = base_df['year'].max()
    mask_mileage_fake = (base_df['mileage'] < min_valid_mileage) & (base_df['year'] < current_year - mileage_grace_years)

    print(f"Пробег > {mileage_upper_bound:,.0f} км ({max_mileage_quantile*100}% квантиль): {mask_mileage_high.sum()} шт.")
    print(f"Подозрительно низкий пробег (авто старше {mileage_grace_years} лет с пробегом < {min_valid_mileage} км): {mask_mileage_fake.sum()} шт.")

    # 3. Аномалии оценки рынка
    safe_imv = base_df['imv'].replace(0, np.nan)
    delta_p = (base_df['price'] - safe_imv) / safe_imv
    mask_overpriced = delta_p > max_overprice_ratio

    print(f"Цена завышена более чем на {max_overprice_ratio * 100}% от IMV: {mask_overpriced.sum()} шт.")

    print("-" * 50)

    print("\n=== ДЕТЕКТОР АНОМАЛИЙ В ЛОГАХ (CNT_DF) ===")

    # Отсекаем мертвые дни, смотрим только на те, где была активность
    active_days = cnt_df[cnt_df['cnt_contacts'] > 0]['cnt_contacts']

    print(f"95.0% активных дней имеют меньше: {active_days.quantile(0.95):.0f} контактов")
    print(f"99.0% активных дней имеют меньше: {active_days.quantile(0.99):.0f} контактов")
    print(f"99.5% активных дней имеют меньше: {active_days.quantile(0.995):.0f} контактов")
    print(f"99.9% активных дней имеют меньше: {active_days.quantile(0.999):.0f} контактов")
    print(f"99.99% активных дней имеют меньше: {active_days.quantile(0.9999):.0f} контактов") # 126 контактов
    print(f"99.995% активных дней имеют меньше: {active_days.quantile(0.99995):.0f} контактов") # 186 контактов

    # берем середину 150 контактов (между 99.99% и 99.995% перцентилями)
    mask_bot_traffic = cnt_df['cnt_contacts'] > bot_traffic_limit

    print(f"Дней с аномальным трафиком (> {bot_traffic_limit} контактов): {mask_bot_traffic.sum()} шт.")


if __name__ == "__main__":
    base_df = pd.read_csv('data/raw/liquidity_base.csv')
    cnt_df = pd.read_csv('data/raw/liquidity_cnt.csv')

    base_df.columns = base_df.columns.str.strip()
    cnt_df.columns = cnt_df.columns.str.strip()

    scan_anomalies(base_df, cnt_df)

=== АНАЛИЗ ПРОПУСКОВ (NaN) В БАЗЕ ===
                Пропуски  Доля (%)
rating            105405     25.53
reviews           105405     25.53
equipment          95499     23.13
imv                69118     16.74
removal_reason     61625     14.93
modification         714      0.17
generation           161      0.04
--------------------------------------------------

=== ДЕТЕКТОР АНОМАЛИЙ В BASE_DF ===
Авто дешевле 50,000 руб: 2097 шт.
Авто дороже 26,604,000 руб (99.9% квантиль): 413 шт.
Пробег > 520,231 км (99.5% квантиль): 2064 шт.
Подозрительно низкий пробег (авто старше 3 лет с пробегом < 100 км): 242 шт.
Цена завышена более чем на 200.0% от IMV: 748 шт.
--------------------------------------------------

=== ДЕТЕКТОР АНОМАЛИЙ В ЛОГАХ (CNT_DF) ===
95.0% активных дней имеют меньше: 8 контактов
99.0% активных дней имеют меньше: 17 контактов
99.5% активных дней имеют меньше: 23 контактов
99.9% активных дней имеют меньше: 45 контактов
99.99% активных дней имеют меньше: 126 контактов
99

### 1. Что делаем с аномалиями (Выбросы)

* Цена < 30к (439 шт): Сносим к чертовой матери. Это металлолом, продажа ПТС или мошенники. 
* Цена > 20 млн (848 шт): Оставляем, раз ты настаиваешь, но перед кластеризацией будем использовать логарифм цены (np.log1p(price)), иначе они утащат на себя центроиды.
* Скрученный пробег (1904 шт): Сносим. Это 0.4% от всей базы (у вас там около 412к строк судя по процентам). Нет смысла тратить время на предсказание их реального пробега, это просто грязный шум от перекупов.
* Неадекватная цена $\Delta P > 2$ (748 шт): Сносим. Чуваки, продающие тачки в 3 раза дороже рынка, не получат контактов просто из-за своей жадности. Это не системная проблема, это клиника.
* Боты в логах (> 500 контактов за день): Мы не удаляем эти дни, иначе порвем временные ряды (ту самую плотную сетку). Мы их «клипаем» (capping) — приравниваем все аномальные значения к 99-му перцентилю (например, к 50 контактам).

### 2. Что делаем с пропусками (NaN)

Здесь нужна жесткая продуктовая логика. Пропуск — это часто не ошибка, а сигнал.

* `rating` и `reviews` (25.53%): Это новые продавцы. У них нет рейтинга. Заполняем reviews нулем, а rating — маркером -1 (или 0). Алгоритм сам поймет, что "минус первый" рейтинг — это каста новичков, и у них ликвидность обычно ниже из-за недоверия.
* `equipment` (23.13%): Огромный кусок. Лень продавца расписывать комплектацию — это шикарная фича. Заменяем NaN на строковое значение "Не указано". Вы потом на защите покажете, что объявления без описания комплектации теряют 30% звонков.
* `imv` (16.74%): Это рыночная оценка, критически важная для расчета выгоды. Выкидывать 16% базы — непозволительная роскошь. Мы заполним пустые imv медианной ценой машин той же марки, модели и года. 
* `removal_reason` (14.93%): Причина снятия. Она вообще не должна участвовать в предсказании ликвидности, потому что на момент публикации объявления мы её не знаем (это заглядывание в будущее, data leak). Заполняем "Неизвестно".
* `modification` и `generation` (<0.2%): Удаляем эти строки. Их слишком мало, чтобы возиться.
* сносим пробег > 99 перцентиля, так как ориентируемся на массу рынку

In [3]:
def run_cleaning_pipeline(
    base_df: pd.DataFrame,
    cnt_df: pd.DataFrame,
    clean_base_path: str = 'data/processed/clean_base.csv',
    clean_cnt_path: str = 'data/processed/clean_cnt.csv',
    min_valid_price: int = 50000,
    max_price_quantile: float = 0.999,
    max_mileage_quantile: float = 0.995,
    min_valid_mileage: int = 100,
    mileage_grace_years: int = 3,
    max_overprice_ratio: float = 2.0,
    bot_traffic_limit: int = 150
) -> None:
    """
    Основной пайплайн очистки данных от мусора, выбросов и API-спама.

    Считывает сырые данные, применяет бизнес-правила и статистические фильтры,
    после чего сохраняет кристально чистую базу для сборки витрин.
    """
    print(f"Исходный размер базы: {base_df.shape[0]}")

    # ==========================================
    # ШАГ 1: УДАЛЕНИЕ МУСОРА И АНОМАЛИЙ
    # ==========================================
    # 1. Цена: сносим металлолом и экстремальный верхний квантиль
    price_upper_bound = base_df['price'].quantile(max_price_quantile)
    base_df = base_df[(base_df['price'] >= min_valid_price) & (base_df['price'] <= price_upper_bound)]

    # 2. Пробег: сносим фейки и экстремальный верхний квантиль (ошибки ввода/износ)
    mileage_upper_bound = base_df['mileage'].quantile(max_mileage_quantile)
    current_year = base_df['year'].max()
    fake_mileage_mask = (base_df['mileage'] < min_valid_mileage) & (base_df['year'] < current_year - mileage_grace_years)

    # Фильтруем по обеим маскам
    base_df = base_df[(base_df['mileage'] <= mileage_upper_bound) & (~fake_mileage_mask)]

    # 3. Сносим строки с микро-пропусками
    base_df = base_df.dropna(subset=['modification', 'generation'])

    # ==========================================
    # ШАГ 2: УМНОЕ ЗАПОЛНЕНИЕ ПРОПУСКОВ (IMPUTATION)
    # ==========================================
    base_df['rating'] = base_df['rating'].fillna(-1.0)
    base_df['reviews'] = base_df['reviews'].fillna(0)
    base_df['equipment'] = base_df['equipment'].fillna('Не указано')
    base_df['removal_reason'] = base_df['removal_reason'].fillna('Неизвестно')

    # Восстановление рыночной оценки (IMV)
    imv_fill = base_df.groupby(['brand', 'model', 'year'])['price'].transform('median')
    base_df['imv'] = base_df['imv'].fillna(imv_fill).fillna(base_df['price'])

    # ==========================================
    # ШАГ 3: ФИЛЬТРАЦИЯ ЖАДНЫХ ПРОДАВЦОВ
    # ==========================================
    base_df['delta_p'] = (base_df['price'] - base_df['imv']) / base_df['imv']
    base_df = base_df[base_df['delta_p'] <= max_overprice_ratio]

    print(f"Размер базы после жесткой чистки: {base_df.shape[0]}")

    # ==========================================
    # ШАГ 4: ЧИСТКА ЛОГОВ (CAPPING)
    # ==========================================
    # Ограничиваем ботов жестким хардкапом, чтобы сохранить органическую дисперсию
    cnt_df['cnt_contacts'] = np.where(
        cnt_df['cnt_contacts'] > bot_traffic_limit,
        bot_traffic_limit,
        cnt_df['cnt_contacts']
    )

    # Синхронизация логов с очищенной базой
    valid_ids = base_df['id'].unique()
    cnt_df = cnt_df[cnt_df['id'].isin(valid_ids)]

    print("Сохранение очищенных данных...")
    base_df.to_csv(clean_base_path, index=False)
    cnt_df.to_csv(clean_cnt_path, index=False)
    print("Готово. Данные кристально чистые.")

if __name__ == "__main__":
    base_df = pd.read_csv('data/raw/liquidity_base.csv')
    cnt_df = pd.read_csv('data/raw/liquidity_cnt.csv')

    run_cleaning_pipeline(base_df, cnt_df)

Исходный размер базы: 412801
Размер базы после жесткой чистки: 406330
Сохранение очищенных данных...
Готово. Данные кристально чистые.


### Методология формирования аналитических витрин

Для решения задач хакатона была разработана архитектура данных, разделенная на два уровня агрегации. Это позволяет корректно учитывать как статические характеристики лотов, так и динамику пользовательского интереса.

#### 1. Статическая витрина (Факторы ликвидности)

Целевая переменная для каждого объявления $i$ (суммарная ликвидность) определяется как интегральный показатель контактов за весь период экспозиции:

$$L_{i} = \sum_{t=1}^{T_{i}} C_{i, t}$$

Где:
- $L_{i}$ — совокупная ликвидность лота;
- $C_{i, t}$ — количество контактов по объявлению в день $t$;
- $T_{i}$ — общее время жизни объявления в системе.

В случае отсутствия лога активности для конкретного $ID$, значение $L_{i}$ принимается равным $0$. Данная витрина используется для выявления ключевых характеристик автомобиля (цена, пробег, регион), влияющих на итоговый спрос.

#### 2. Динамическая витрина (Анализ временных рядов и VAS)

Для поиска оптимального момента подключения платных услуг (VAS) необходимо анализировать форму кривой затухания спроса. Чтобы избежать смещения оценок из-за "пропусков" в данных (дней без контактов), произведено развертывание векторов времени в плотную сетку $\mathbf{T}_{i} = \{1, 2, \dots, T_{i}\}$. 

Функция ежедневной ликвидности $f(t)$ для объявления $i$ формализована следующим образом:

$$f(t) = 
\begin{cases} 
x_{t}, & \text{при наличии записи в логе в день } t \\
0, & \text{в противном случае}
\end{cases}$$

Такой подход позволяет корректно рассчитывать производную функции спроса и определять точку $t_{opt}$, в которой падение органического интереса требует применения инструментов продвижения.

### отсекаем 10 перцентиль, чтобы не оценивать редкие модели, берем фокус на самых популярных моделях

In [4]:
# 1. Загружаем уже первично очищенные данные
base_df = pd.read_csv('data/processed/clean_base.csv')
cnt_df = pd.read_csv('data/processed/clean_cnt.csv')

print(f"Размер базы ДО среза марок: {base_df.shape[0]} строк")
print(f"Потребление памяти ДО: {base_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# ==========================================
# ШАГ 1: ФОКУС НА МАСС-МАРКЕТ (Scope Limitation)
# ==========================================
# Считаем частоту каждой марки
brand_counts = base_df['brand'].value_counts()

# Берем порог 90-го перцентиля. 
# Это значит, что мы отсекаем 90% марок, которые встречаются реже всего (длинный хвост),
# и оставляем топ-10% марок, которые генерируют основной объем рынка.
threshold = brand_counts.quantile(0.90)

# Список ходовых марок
top_brands = brand_counts[brand_counts >= threshold].index

# Фильтруем базу
base_df = base_df[base_df['brand'].isin(top_brands)]

# Обязательно чистим логи, чтобы там не остались контакты от удаленных редких тачек
valid_ids = base_df['id'].unique()
cnt_df = cnt_df[cnt_df['id'].isin(valid_ids)]

# ==========================================
# ШАГ 2: ОПТИМИЗАЦИЯ ПАМЯТИ (Тип Category)
# ==========================================
# Находим все колонки со строковым типом (object)
object_cols = base_df.select_dtypes(include=['object']).columns

# Конвертируем в category
for col in object_cols:
    base_df[col] = base_df[col].astype('category')

print("-" * 40)
print(f"Размер базы ПОСЛЕ среза марок: {base_df.shape[0]} строк")
print(f"Потребление памяти ПОСЛЕ: {base_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Перезаписываем файлы. Теперь они идеальны для сборки витрин.
base_df.to_csv('data/processed/clean_base.csv', index=False)
cnt_df.to_csv('data/processed/clean_cnt.csv', index=False)

Размер базы ДО среза марок: 406330 строк
Потребление памяти ДО: 499.99 MB
----------------------------------------
Размер базы ПОСЛЕ среза марок: 358011 строк
Потребление памяти ПОСЛЕ: 88.37 MB


In [5]:
def build_static_mart(base_df: pd.DataFrame, cnt_df: pd.DataFrame) -> pd.DataFrame:
    """
    Собирает статическую витрину: 1 объявление = 1 строка с итоговым количеством контактов.
    
    Args:
        base_df: Очищенный датафрейм с характеристиками авто.
        cnt_df: Очищенный датафрейм с логами контактов.
        
    Returns:
        pd.DataFrame: Готовая витрина для обучения моделей (LightGBM и др.).
    """
    total_contacts = cnt_df.groupby('id', as_index=False)['cnt_contacts'].sum()
    total_contacts.rename(columns={'cnt_contacts': 'total_contacts'}, inplace=True)

    df_task1 = base_df.merge(total_contacts, on='id', how='left')
    df_task1['total_contacts'] = df_task1['total_contacts'].fillna(0)
    
    return df_task1


def build_dynamic_mart(base_df: pd.DataFrame, cnt_df: pd.DataFrame) -> pd.DataFrame:
    """
    Собирает динамическую витрину с плотной временной сеткой (векторизованный метод).
    
    Args:
        base_df: Очищенный датафрейм с характеристиками авто.
        cnt_df: Очищенный датафрейм с логами контактов.
        
    Returns:
        pd.DataFrame: Временные ряды (Time-to-Sell) без пропусков по дням.
    """
    # 1. Расчет времени жизни с использованием векторизованного clip
    max_date = base_df['close_time'].max()
    close_filled = base_df['close_time'].fillna(max_date)
    
    lifetime_days = (close_filled - base_df['start_time']).dt.days
    base_df['lifetime_days'] = lifetime_days.fillna(1).clip(lower=1).astype(int)

    # 2. Векторизованная генерация плотной сетки (БЕЗ ЦИКЛОВ)
    # Размножаем индексы пропорционально lifetime_days
    grid = base_df[['id', 'lifetime_days']].loc[base_df.index.repeat(base_df['lifetime_days'])].reset_index(drop=True)
    
    # Восстанавливаем номер дня: кумулятивный счетчик внутри каждого id
    grid['day'] = grid.groupby('id').cumcount() + 1
    grid = grid.drop(columns=['lifetime_days'])

    # 3. Мерж с логами и заполнение дней без контактов нулями
    df_dynamics = grid.merge(cnt_df, on=['id', 'day'], how='left')
    df_dynamics['cnt_contacts'] = df_dynamics['cnt_contacts'].fillna(0)

    # 4. Финальное обогащение характеристиками авто
    df_task2 = df_dynamics.merge(
        base_df.drop(columns=['lifetime_days']), 
        on='id', 
        how='inner'
    )
    
    return df_task2


def main():
    """Основной пайплайн сборки витрин."""
    print("Загрузка очищенных данных")
    base_df = pd.read_csv('data/processed/clean_base.csv')
    base_df['start_time'] = pd.to_datetime(base_df['start_time'], errors='coerce')
    base_df['close_time'] = pd.to_datetime(base_df['close_time'], errors='coerce')
    
    cnt_df = pd.read_csv('data/processed/clean_cnt.csv')

    print("Сборка статической витрины (Task 1)")
    df_static = build_static_mart(base_df, cnt_df)
    df_static.to_csv('data/processed/processed_task1_total_liquidity.csv', index=False)
    print(f"Готово. Размер: {df_static.shape}")

    print("Сборка плотной динамической витрины (Task 2 & 3)")
    df_dynamic = build_dynamic_mart(base_df, cnt_df)
    df_dynamic.to_csv('data/processed/processed_task2_dense_dynamics.csv', index=False)
    print(f"Готово. Размер: {df_dynamic.shape}")

if __name__ == "__main__":
    main()

Загрузка очищенных данных
Сборка статической витрины (Task 1)
Готово. Размер: (358011, 25)
Сборка плотной динамической витрины (Task 2 & 3)
Готово. Размер: (7618616, 27)


In [7]:
import pandas as pd
import numpy as np


# Настройка строгого стиля для презентации
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")

# Загружаем собранные витрины
df_stat = pd.read_csv('data/processed/processed_task1_total_liquidity.csv')
df_dyn = pd.read_csv('data/processed/processed_task2_dense_dynamics.csv')

# ==========================================
# 1. КОРРЕЛЯЦИОННАЯ МАТРИЦА (Критерий жюри)
# ==========================================
plt.figure(figsize=(10, 8))
numeric_cols = ['price', 'imv', 'delta_p', 'mileage', 'year', 'total_contacts']
corr_matrix = df_stat[numeric_cols].corr(method='spearman')
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title('График 1: Ранговая корреляция факторов ликвидности')
plt.tight_layout()
plt.savefig('data/processed/viz_1_correlation.png')
plt.close()

# ==========================================
# 2. ВЛИЯНИЕ ЖАДНОСТИ ПРОДАВЦА (Scatter plot)
# ==========================================
# Показываем, как переоценка (delta_p) убивает звонки
plt.figure(figsize=(10, 6))
# Берем случайные 10k точек, чтобы не перегружать график
sample_df = df_stat.sample(min(10000, len(df_stat)), random_state=42)
sns.scatterplot(x='delta_p', y='total_contacts', data=sample_df, alpha=0.3, edgecolor=None)
plt.axvline(0, color='red', linestyle='--', label='Рыночная цена (IMV)')
plt.title('График 2: Падение ликвидности при завышении цены (Overprice)')
plt.xlabel('Отклонение от рынка (delta_p)')
plt.ylabel('Суммарные контакты')
plt.legend()
plt.tight_layout()
plt.savefig('data/processed/viz_2_overprice.png')
plt.close()

# ==========================================
# 3. ПОИСК СЕГМЕНТОВ (Boxplot)
# ==========================================
# Для примера разобьем машины на 3 базовых сегмента по возрасту
df_stat['age_segment'] = pd.cut(
    df_stat['year'], 
    bins=[1900, 2010, 2019, 2025], 
    labels=['Старые (>15 лет)', 'Средние (5-15 лет)', 'Свежие (<5 лет)']
)

plt.figure(figsize=(10, 6))
sns.boxplot(x='age_segment', y='total_contacts', data=df_stat, showfliers=False)
plt.title('График 3: Сравнение ликвидности по возрастным сегментам')
plt.ylabel('Контакты (без выбросов)')
plt.tight_layout()
plt.savefig('data/processed/viz_3_segments_boxplot.png')
plt.close()

# ==========================================
# 4. ТИПЫ КРИВЫХ НАКОПЛЕНИЯ КОНТАКТОВ (Критерий жюри)
# ==========================================
# Мержим сегменты в динамическую витрину для графика
df_dyn = df_dyn.merge(df_stat[['id', 'age_segment']], on='id', how='left')

plt.figure(figsize=(12, 6))
# Считаем среднее количество контактов в каждый день для каждого сегмента
avg_curves = df_dyn.groupby(['age_segment', 'day'])['cnt_contacts'].mean().reset_index()
sns.lineplot(x='day', y='cnt_contacts', hue='age_segment', data=avg_curves[avg_curves['day'] <= 20])
plt.title('График 4: Профили затухания интереса (Time-to-Sell)')
plt.xlabel('День жизни объявления')
plt.ylabel('Среднее кол-во контактов')
plt.xlim(1, 20)
plt.tight_layout()
plt.savefig('data/processed/viz_4_time_curves.png')
plt.close()

# ==========================================
# 5. СРАВНЕНИЕ ТОП-МАРОК МАСС-МАРКЕТА
# ==========================================
plt.figure(figsize=(12, 6))
top_20_brands = df_stat['brand'].value_counts().nlargest(20).index
df_top_brands = df_stat[df_stat['brand'].isin(top_20_brands)]
sns.barplot(x='brand', y='total_contacts', data=df_top_brands, estimator=np.median, errorbar=None)
plt.title('График 5: Медианная ликвидность Топ марок масс-маркета')
plt.ylabel('Медиана контактов')
plt.tight_layout()
plt.savefig('data/processed/viz_5_top_brands.png')
plt.close()

print("EDA завершен. 5 графиков сохранены в data/processed/")

EDA завершен. 5 графиков сохранены в data/processed/
